In [1]:
# ===================== MODIFICACIONES μ =====================
import numpy as np

def mu_fun(lambda_val, phi0):
    # TODO: reemplazar por ecuación (3.6) Kang & Juang (2010)
    return - (lambda_val - 1.0) * phi0

def Solve_incremental_mu(phi0, lambda_target=2.0, n_steps=16):
    lambdas = np.linspace(1.1, lambda_target, n_steps)
    mus = [mu_fun(l, phi0) for l in lambdas]

    results = []
    for l, mu in zip(lambdas, mus):
        print(f"Resolviendo para lambda={l:.3f}, mu={mu:.5f}")
        # IMPORTANTE:
        # Aquí debes reemplazar la lógica donde antes variabas gamma:
        # gel.gamma = constante
        # gel.mu_bar = mu
        # SolveNonlinearMinProblem(...)
        results.append((l, mu))

    return results
# ============================================================


from ngsolve import *
from ngsolve.webgui import Draw
import numpy as np
import scipy.optimize
import datetime
import sys

class gel_3D:
    def __init__(self, length=90.0, width =15.0, thickness=1.6, phi0=0.2, mu_bar=-0.05):
        self.phi0 = phi0
        print(f'Initial polymer volume fraction = {phi0:.2f} [dimensionless];', end=' ')
        self.mu_bar = mu_bar   # 🔥 NUEVO
        print(f'Normalized chemical potential = {mu_bar:.6f} [dimensionless?];', end=' ')
        #
        # Values for T and V_m taken from p.1584 in Kang & Huang, JMPS 58 (2010)
        T = 25+273.15   # 25ºC, in [K]
        K_B = 1.380649e-23  # in m^2*kg*s^{-2}*K^{-1}, i.e. in N*m/K
        V_m = 3e-29     # volume of a molecule of solvent, in this case water, in m^3
        # self.entropic_unit = 136.6  # measured in MPa
        self.entropic_unit = K_B*T/V_m*1e-6  # measured in MPa
        print(f'Entropic unit = {self.entropic_unit:.2f} [MPa];', end='\n')
        #
        # self.G = 0.13               # measured in MPa
        # self.gamma = self.G/self.entropic_unit
        self.gamma = 0.001           # As in the simulations of Sect. 5 in Kang & Huang JMPS 2010
        # self.chi =  0.348
        self.chi = 0.4
        self.G = self.gamma*self.entropic_unit
        print(f'gamma=N*V_m = {self.gamma:.2e} [dimensionless];', end=' ')
        print(f'Shear modulus = N*K_B*T = {self.G:.2f} [MPa];', end=' ')
        print(f'Flory parameter = {self.chi:.3f} [dimensionless];', end='\n')
        #
        vapor_pressure = 3.2e-3    # 3.2 KPa, but measured in MPa
        self.p0_bar = vapor_pressure/(self.entropic_unit)
        print(f'Normalized vapor pressure = {self.p0_bar:.2e} [dimensionless];', end=' ')
        self.p_bar = self.p0_bar * np.exp(mu_bar)
        print(f'External solvent pressure = {self.p_bar*self.entropic_unit*1e3:.2f} [KPa];', end=' ')
        print(f'Normalized external pressure = {self.p_bar:.2e} [MPa];', end='\n')
        #        
        # self.density =  1.23 # measured in [g/mL]

        self.L = length      # measured in mm
        self.d = thickness    # measured in mm
        self.w = width # measured in mm

        self.filename_suffix = f'_phi0={self.phi0:.1f}_muBarAbs={np.abs(mu_bar):.6f}'
        print(f'Filename suffix: ' + self.filename_suffix, end='\n')

        def auxIsotropic(s):
            return s*self.dH(s*s*s) + self.gamma
        max_attempts = 100000
        attempts = 0
        lambda_initial = phi0*1.1
        while attempts< max_attempts:
            aux_value = auxIsotropic(lambda_initial)
            if aux_value>0:
                break
            lambda_initial+=0.01
            attempts+=1
        self.lambda_iso = scipy.optimize.fsolve(auxIsotropic, lambda_initial)[0]
        print(f'Isotropic extension: {self.lambda_iso:.3f}; lambda_initial = {lambda_initial}', end=' ')

        def auxUniaxial(s):
            return s*self.gamma + self.dH(s)
        max_attempts = 100000
        attempts = 0
        lambda_initial = phi0*1.1
        while attempts< max_attempts:
            aux_value = auxUniaxial(lambda_initial)
            if aux_value>0:
                break
            lambda_initial+=0.01
            attempts+=1
        self.lambda_target = scipy.optimize.fsolve(auxUniaxial, lambda_initial)[0]
        print(f'Uniaxial extension: {self.lambda_target:.3f}; lambda_initial = {lambda_initial}', end='\n')

        def auxEnergyDensity(lambda1, lambda2, lambda3):
            gel=self; phi0=gel.phi0; G=gel.G; chi=gel.chi; nu=gel.entropic_unit; gamma=gel.gamma; mu_bar=gel.mu_bar; p_bar=gel.p_bar
            J= lambda1*lambda2*lambda3
            phi = phi0/J
            return 0.5*G*(lambda1**2 + lambda2**2 + lambda3**2 - 3) + nu*((J-phi0)*np.log(1-phi) + phi0*chi*(1-phi) - gamma*log(J) + (p_bar - mu_bar)*(J-phi0) )

        lambda_iso = self.lambda_iso
        self.reference_energy_density = auxEnergyDensity(lambda_iso, lambda_iso, lambda_iso)                
        print(f'Energy density of isotropic expansion: {self.reference_energy_density:.5f}', end=' ')

    def phi(self, J):
        return self.phi0/J

    def H(self, J):
        return (J - self.phi0)*log(1-self.phi(J))  + self.phi0 * self.chi*(1-self.phi(J)) - self.gamma*log(J) + (self.p_bar - self.mu_bar)*(J-self.phi0)

    def dH(self, J):
        return self.phi(J) + np.log(1-self.phi(J)) + self.chi * self.phi(J)**2  - self.gamma/J +self.p_bar - self.mu_bar

    def Gfun(self, lamb):
        nu = self.entropic_unit
        return (-self.dH(lamb)/lamb)*nu
        
    # energy density in [MPa]
    def W(self, F):
               
        J = Det(F)
        C_tensor = F.trans * F
        
        gel = self
        G = gel.G
        nu = gel.entropic_unit               
        
        reference_energy_density =  self.reference_energy_density
        
        return 0.5*G*(Trace(C_tensor) - 3) + nu*gel.H(J) - reference_energy_density

class Solve_gel3d:
    def __init__(self, gel, order=1):         # corner_refinement capability not yet activated
        self.gel = gel
        self.order = order
        self.start_time = datetime.datetime.now()  

    def add_mesh(self, mesh_file):        
        self.mesh = Mesh(mesh_file)
    
    def Space(self):
        # Finite element space with zero-displacement boundary condition 
        # on all of the bottom interface
        self.fes = VectorH1(self.mesh, order=self.order, dirichlet="bonded|debonded")
        print('nDoF = {}'.format(self.fes.ndof))
        
    def model(self):
        u  = self.fes.TrialFunction()
        I = Id(self.mesh.dim)
        F = I + Grad(u)

        # gravity = CoefficientFunction((0,0,-9.8))
        # gel_density_in_g_per_mL = self.gel.density

        def negpart(var):
            return (sqrt(var**2)-var)*0.5        
        
        AA = 1e5

        # hydrogel model        
        self.a = BilinearForm(self.fes, symmetric=False)

        # 🔥 AQUÍ ya entra μ automáticamente desde gel.W(F)
        self.a += Variation(self.gel.W(F).Compile() * dx)

        # # gravedad
        # self.a += Variation( -((1e-6)*gel_density_in_g_per_mL)*InnerProduct(gravity, u)*dx )

        # contacto
        self.a += Variation(AA*negpart(y+u[1])**2 * dx)
        
    def Solve_incremental_softening(self):
        self.Space()
        self.gfu = GridFunction(self.fes)
        self.gfu.vec[:] = 0
        
        lambda_initial = 1.1
        G_end = self.gel.G
        G0 = self.gel.Gfun(lambda_initial)

        nIterations = 15
        self.gel.G = Parameter(G0)

        self.model()
        
        lambda_list = np.linspace(lambda_initial, self.gel.lambda_target, nIterations, endpoint=False)
        G_list = [self.gel.Gfun(la) for la in lambda_list]
        
        G_list.append(G_end)
              
        filename='gridfunctions/result'+ \
            self.gel.filename_suffix+ \
            "_order={}".format(self.order)

        tol=1e-3
        maxits=100
        
        #indexes_iterations = range(nIterations+1)
        indexes_iterations = [0]

        for numIteration in indexes_iterations:
            G_i = G_list[numIteration]
            print("*** Iteration #", numIteration, ". Shear modulus G = ", G_i)

            if numIteration == nIterations:
                tol = 1e-6
                maxits = 500
            
            self.gel.G.Set(G_i)

            self.gfu, _, _ = SolveNonlinearMinProblem(
                a=self.a,
                gfu=self.gfu,
                maxits=maxits,
                tol=tol,
                alpha=1e-2
            )

            self.gfu.Save(filename + '_iter=' + str(numIteration).zfill(2) + '.gfu')

            print("Total time elapsed =" + str(datetime.datetime.now() - self.start_time))

# experimental acceleration
def SolveNonlinearMinProblem(a, gfu, tol=1e-08, maxits=50, alpha=5e-2):
    
    start_time = datetime.datetime.now()  

    res = gfu.vec.CreateVector()
    du  = gfu.vec.CreateVector()
    
    # precond local, multigrid, bddc
    precond = 'bddc'
    c = Preconditioner(a, precond)

    for it in range(maxits):

        with TaskManager():
            a.Apply(gfu.vec, res)
            c = Preconditioner(a, precond)
            a.AssembleLinearization(gfu.vec)
            
            c.Update()
            inv = CGSolver(a.mat, c.mat, maxsteps=1000)

            du.data = alpha * inv * res

            # update
            gfu.vec.data -= du

        # stopping criteria
        stopcritval = sqrt(abs(InnerProduct(du, res)))

        print("Newton iteration:", it, "Time elapsed =" + str(datetime.datetime.now() - start_time))
        print("<A u", it, ", A u", it, ">_{-1}^0.5 = ", stopcritval)

        if stopcritval < tol:
            break

    return gfu, stopcritval, it


# data = [dummy, L, w, d, phi0, abs(mu_bar)]   We will suppose that mu_bar is negative
data = ['','90', '15.0', '1.62', '1', '0.0916'] 
order = 1

mesh_file = 'meshes/mesh0.vol.gz'

L = float(data[1])
d = float(data[2])
w = float(data[3])
phi0 = float(data[4])
mu_bar = - float(data[5])

print(f'L={L}, w={w}, d={d}, phi0={phi0}, mu_bar={mu_bar}')

gel = gel_3D(length=L, width=w, thickness=d, phi0=phi0, mu_bar=mu_bar)

modelling = Solve_gel3d(gel, order=order)
modelling.add_mesh(mesh_file)
modelling.Space()

modelling = Solve_gel3d (gel, order=order)
modelling.add_mesh(mesh_file)
modelling.Solve_incremental_softening()

L=90.0, w=1.62, d=15.0, phi0=1.0, mu_bar=-0.0916
Initial polymer volume fraction = 1.00 [dimensionless]; Normalized chemical potential = -0.091600 [dimensionless?]; Entropic unit = 137.21 [MPa];
gamma=N*V_m = 1.00e-03 [dimensionless]; Shear modulus = N*K_B*T = 0.14 [MPa]; Flory parameter = 0.400 [dimensionless];
Normalized vapor pressure = 2.33e-05 [dimensionless]; External solvent pressure = 2.92 [KPa]; Normalized external pressure = 2.13e-05 [MPa];
Filename suffix: _phi0=1.0_muBarAbs=0.091600
Isotropic extension: 1.262; lambda_initial = 1.2700000000000002 Uniaxial extension: 2.000; lambda_initial = 2.0100000000000007
Energy density of isotropic expansion: -55.06968 nDoF = 90978
nDoF = 90978
*** Iteration # 0 . Shear modulus G =  133.16116878070778
Newton iteration: 0 Time elapsed =0:00:01.570044
<A u 0 , A u 0 >_{-1}^0.5 =  nan
Newton iteration: 1 Time elapsed =0:00:03.041005
<A u 1 , A u 1 >_{-1}^0.5 =  nan
Newton iteration: 2 Time elapsed =0:00:04.475834
<A u 2 , A u 2 >_{-1}^0.5 =